In [1]:
import numpy as np
from numba import cuda
from tensorflow.keras.datasets import cifar10

# CUDA Kernel for 2x2 Max Pooling
@cuda.jit
def max_pooling_kernel(input_img, output_img):
    row, col = cuda.grid(2)

    if row < output_img.shape[0] and col < output_img.shape[1]:

        start_row = row * 2
        start_col = col * 2

        max_val = input_img[start_row, start_col]

        for i in range(2):
            for j in range(2):
                if input_img[start_row + i, start_col + j] > max_val:
                    max_val = input_img[start_row + i, start_col + j]

        output_img[row, col] = max_val


# Load CIFAR-10 Dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Select first image
image = x_train[0]

# Convert RGB image to grayscale
gray_image = np.mean(image, axis=2).astype(np.float32)

print("Original Image Shape:", gray_image.shape)

# Output size after 2x2 pooling
pooled_rows = gray_image.shape[0] // 2
pooled_cols = gray_image.shape[1] // 2

pooled_image = np.zeros((pooled_rows, pooled_cols), dtype=np.float32)

# Copy to GPU
d_input = cuda.to_device(gray_image)
d_output = cuda.to_device(pooled_image)

# Thread configuration
threads_per_block = (16, 16)

blocks_per_grid_x = (pooled_rows + threads_per_block[0] - 1) // threads_per_block[0]
blocks_per_grid_y = (pooled_cols + threads_per_block[1] - 1) // threads_per_block[1]

blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)

# Launch kernel
max_pooling_kernel[blocks_per_grid, threads_per_block](
    d_input,
    d_output
)

# Copy result back
pooled_image = d_output.copy_to_host()

print("Pooled Image Shape:", pooled_image.shape)

print("\nSample Output:")
print(pooled_image[:5, :5])

2026-06-01 07:08:21.218697: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780297701.403183      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780297701.457497      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780297701.896204      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780297701.896249      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780297701.896252      58 computation_placer.cc:177] computation placer alr

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Original Image Shape: (32, 32)


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:748: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Pooled Image Shape: (16, 16)

Sample Output:
[[ 61.333332  54.666668  91.       111.666664 120.666664]
 [ 25.        65.666664  91.666664  92.666664  83.333336]
 [ 53.666668  83.        89.666664  85.333336  87.333336]
 [ 82.        96.666664  89.666664  91.666664  95.333336]
 [121.       117.333336  88.333336  90.        99.      ]]


In [2]:
from numba import cuda

print(cuda.is_available())

True
